In [2]:
import gradio as gr
def greet(name):
  return "Hello " + name + "!"

demo = gr.Interface(fn=greet, inputs="text", outputs="text")
demo.launch()

c:\Users\ABHCST9\Desktop\dev\workspace\next_ai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [3]:
import torch
import torchvision.models as models
from torchvision.models import ResNet50_Weights

# 사전학습 모델 로드 (torch.hub 안 씀)
weights = ResNet50_Weights.IMAGENET1K_V2
# eval: 추가 학습을 막기
model = models.resnet50(weights=weights).eval()

# 라벨도 weights에서 바로 가져옴 (requests로 다운받을 필요 없음)
labels = weights.meta["categories"]

# 전처리도 weights에 내장되어 있음
preprocess = weights.transforms()

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\ABHCST9/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:08<00:00, 11.7MB/s]


In [4]:
# 주어진 이미지에 대한 예측
def predict(inp):
  # unsqueeze 0번째 축에 []로 감싸기(model이 원하는 형태이므로)
  inp = preprocess(inp).unsqueeze(0)

  with torch.no_grad():
    prediction = torch.nn.functional.softmax(model(inp)[0], dim=0)
    confidences = {labels[i]: float(prediction[i]) for i in range(len(labels))}
  return confidences

In [5]:
import requests
import os

# 이미지를 다운로드
def download_image(url, save_path):
  headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
  }
  try:
    response = requests.get(url, stream=True, headers=headers, verify=False)
    response.raise_for_status()
    with open(save_path, 'wb') as file:
      for chunk in response.iter_content(chunk_size=8192):
        file.write(chunk)
  except requests.exceptions.RequestException as e:
    print(f"오류 발생: {e}")
# 예제 이미지 URL
file_path = {
    "lion.jpg": "https://upload.wikimedia.org/wikipedia/commons/7/73/Lion_waiting_in_Namibia.jpg",
    "plane.jpg": "https://upload.wikimedia.org/wikipedia/commons/e/e5/Airbus_A350-941_F-WZGG_MSN002_ZWS_2018-02-14.jpg"
}
# 예제 이미지 다운로드
for i in file_path:
    download_image(file_path[i], i)

c:\Users\ABHCST9\Desktop\dev\workspace\next_ai\venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'upload.wikimedia.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\ABHCST9\Desktop\dev\workspace\next_ai\venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'upload.wikimedia.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


오류 발생: 404 Client Error: Not Found for url: https://upload.wikimedia.org/wikipedia/commons/e/e5/Airbus_A350-941_F-WZGG_MSN002_ZWS_2018-02-14.jpg


In [6]:
import gradio as gr

# Gradio interface 실행
gr.Interface(
  fn=predict,
  inputs=gr.Image(type="pil"),
  outputs=gr.Label(num_top_classes=3),
  examples=["lion.jpg", "plane.jpg"]
).launch(share=True)


* Running on local URL:  http://127.0.0.1:7861

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

# 데이터 전처리를 위한 변환
# Compose는 transforms의 처리를 하나로 묶어줌
# 기존의 형태
# img = transforms.Resize((224, 224))(img)
# img = transforms.ToTensor()(img)
# img = transforms.Normalize(mean=[...], std=[...])(img)

transform = transforms.Compose([
  transforms.Resize((224, 224)), # 입력 이미지의 크기를 강제 변환
  transforms.ToTensor(), # tensor화
  transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # 표준화
])

# # 훈련 dataset load
# train_dataset = datasets.ImageFolder(root='./train', transform=transform)
# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# # test dataset load
# test_dataset = datasets.ImageFolder(root='./test', transform=transform)
# test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

# model = models.resnet50(pretrained=True)
# num_ftrs = model.fn.in_features
# model.fc = nn.Linear(num_ftrs, 2)

# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.parameters(), lr=0.00005)

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)

FileNotFoundError: [WinError 3] 지정된 경로를 찾을 수 없습니다: './train'